[← 07 - Window Functions, Part 2](<07-window-functions-part-2.ipynb>) · [Course Overview](<00-course-overview.ipynb>)

# 08a - Changing Data, Part 1: INSERT, UPDATE, DELETE, and the WHERE you can't forget

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samietex/SQL_Tutorial/blob/main/08a-changing-data-part-1.ipynb)

`07 - Window Functions, Part 2` closed on a question it didn't answer: every query since `04` has read from the `orders` table, but where did its rows come from? The honest answer is that you put them there yourself. The setup cell at the top of every notebook since `04`, the one you were told to run once and then ignore, is made of the statements this notebook teaches. This notebook explains them, shows how to change rows after they exist, and shows how one missing line changes every row at once. `08b` then covers the other half: creating tables and the rules they enforce.

*New here? The [Course Overview](<00-course-overview.ipynb>) explains how to run these notebooks. Viewing this on GitHub shows a preview with saved results; it won't execute anything.*

> **By the end of this notebook you'll be able to:** add rows with `INSERT`, change them with `UPDATE`, and remove them with `DELETE`, check exactly which rows a write will touch before running it, use `BEGIN` and `ROLLBACK` to try a change safely, and recognize the orphaned row a careless `DELETE` leaves behind.

> **Read the full lesson:** this notebook is the hands-on companion to [Lesson 8 of the All About Data & More curriculum](https://aaboutdata.substack.com/p/changing-data-in-sql-where-your-rows), which walks through the reasoning in more depth.

## 0. Set up

Run the cell below once each time you open the notebook. After it, any cell that starts with `%%sql` runs SQLite against a small lesson database that lives in memory, so there's nothing to install, and any cell that starts with `%%tsql` runs against SQL Server and AdventureWorks, which only section 9 uses and which has its own one-step setup there.

In [1]:
# Run this cell first. It downloads the small helper that lets the cells below run SQL,
# fetching a fresh copy every time so you always have the latest version.
import os, sys, tempfile, urllib.request
helper_dir = os.path.join(tempfile.gettempdir(), 'sql_tutorial_helper')
os.makedirs(helper_dir, exist_ok=True)
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/samietex/SQL_Tutorial/assets/sql_tutorial.py',
    os.path.join(helper_dir, 'sql_tutorial.py'),
)
sys.path.insert(0, helper_dir)
%reload_ext sql_tutorial

Ready. %%sql runs SQLite 3.50.4 in memory, %%tsql runs SQL Server.


## 1. Where the rows came from

Here's the setup cell again, unchanged since `05`. It's made of two kinds of statement you're about to learn: `CREATE TABLE`, which makes an empty table, and `INSERT`, which puts rows into it. Run it, and then run the cells below in order, since each one builds on the table the one before it left behind:

In [2]:
%%sql
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    credit_limit INTEGER NOT NULL
);
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    amount INTEGER NOT NULL
);
INSERT INTO customers VALUES
    (1, 'Ada', 100),
    (2, 'Bola', 200),
    (3, 'Chidi', 300),
    (4, 'Dami', 200);
INSERT INTO orders VALUES
    (101, 1, 40),
    (102, 1, 60),
    (103, 2, 80),
    (104, 2, 90),
    (105, 3, 140);

4 rows inserted
5 rows inserted


## 2. Reading leaves the table alone, and writing doesn't

Every query you wrote from `02` to `07` was a `SELECT`, and a `SELECT` has one property worth saying out loud now that it's about to stop being true: it builds a new result and leaves the table exactly as it found it. You could run `07`'s `LAST_VALUE()` query a thousand times and `orders` would still hold the same five rows.

The statements in this notebook work the other way around. `INSERT`, `UPDATE`, and `DELETE` change the rows stored in a table, and instead of handing back a result they report how many rows they changed (the helper prints that count under each cell, and in a plain SQLite session you'd run `SELECT changes();` straight afterwards). `CREATE TABLE` and `ALTER TABLE` work one level up, on the structure of a table rather than its rows. You'll see the two groups called **DML** (data manipulation language) and **DDL** (data definition language) in documentation and job descriptions.

<p align="center">
  <img src="graphics/08a_read_vs_write.png" width="720" alt="SELECT hands back a result set and leaves the table unchanged; INSERT, UPDATE and DELETE hand back a count and change rows for good once they commit; ALTER TABLE hands back nothing and changes the table structure">
</p>

The part that changes how you work is that a write is permanent once it finishes. The [SQLite transaction documentation](https://www.sqlite.org/lang_transaction.html) says a statement run on its own starts its own transaction and commits it "when the last SQL statement finishes", which means there's no undo button waiting for you afterwards. A wrong `SELECT` costs you a confusing result, while a wrong `UPDATE` costs you the data.

## 3. INSERT: adding rows

The form of `INSERT` worth making a habit is the table name, then the columns you're filling in parentheses, then the values in the same order. Here's a fifth customer, Efe:

In [3]:
%%sql
INSERT INTO customers (
    customer_id,
    customer_name,
    credit_limit
)
VALUES (5, 'Efe', 150);

1 row inserted


In [4]:
%%sql
SELECT
    customer_id,
    customer_name,
    credit_limit
FROM customers;

customer_id,customer_name,credit_limit
1,Ada,100
2,Bola,200
3,Chidi,300
4,Dami,200
5,Efe,150


"1 row inserted", and five customers. Notice that the setup cell didn't name its columns; it went straight from `INSERT INTO customers` to `VALUES`, which works only because the values then have to fill every column, in the order the table defines them. That's fine in a throwaway script that created the table two lines earlier, and it's the habit behind a mistake `08b` shows you. The setup cell also shows that one `VALUES` can hold several rows, separated by commas.

## 4. UPDATE: changing rows that already exist

`UPDATE` names a table, `SET` says what the new value is, and `WHERE` says which rows get it. Before running one, run a `SELECT` with the exact same `WHERE` to see the rows you're about to change:

In [5]:
%%sql
SELECT
    customer_id,
    customer_name,
    credit_limit
FROM customers
WHERE customer_id = 2;

customer_id,customer_name,credit_limit
2,Bola,200


One row, Bola with a limit of 200, so the `WHERE` is aiming where you meant it to. Now raise her limit by 50:

In [6]:
%%sql
UPDATE customers
SET credit_limit = credit_limit + 50
WHERE customer_id = 2;

1 row updated


"1 row updated", and Bola's limit is now 250. `SET` reads the row's current value on the right-hand side and writes the new one on the left, so `credit_limit + 50` means "whatever it was, plus 50", not a fixed number.

### Edge case: a WHERE that matches nothing

Here's the same update with a misspelled name:

In [7]:
%%sql
UPDATE customers
SET credit_limit = credit_limit + 50
WHERE customer_name = 'Bolla';

0 rows updated


"0 rows updated", and no error. The [UPDATE documentation](https://www.sqlite.org/lang_update.html) treats this as a perfectly good statement that happened to change nothing, which is why you read the count after every write instead of assuming it worked.

## 5. Common mistake: UPDATE or DELETE without a WHERE

The same documentation spells out the most expensive line in this notebook: "If the UPDATE statement does not have a WHERE clause, all rows in the table are modified by the UPDATE." Leave out the `WHERE` from the statement above and every customer gets 50 more, not just Bola. Nothing warns you, because nothing in the statement says you meant one row, and it's valid SQL.

You can watch this happen without damaging anything, because a transaction you start yourself doesn't commit until you say so. `BEGIN` opens one, and `ROLLBACK` throws away every change made since:

In [8]:
%%sql
BEGIN;

UPDATE customers
SET credit_limit = credit_limit + 50;

SELECT
    customer_name,
    credit_limit
FROM customers
ORDER BY
    customer_id;

ROLLBACK;

5 rows updated


customer_name,credit_limit
Ada,150
Bola,300
Chidi,350
Dami,250
Efe,200


"5 rows updated", and the table shows the damage: Ada at 150, Bola at 300, everyone up by 50. Then `ROLLBACK` undid it. Check:

In [9]:
%%sql
SELECT
    customer_name,
    credit_limit
FROM customers
ORDER BY
    customer_id;

customer_name,credit_limit
Ada,100
Bola,250
Chidi,300
Dami,200
Efe,150


Ada at 100, Bola at 250, Chidi at 300, Dami at 200, and Efe at 150 again, as if the update never ran. Had the cell ended in `COMMIT` instead, the change would have been kept, and outside a transaction there's nothing to roll back to.

<p align="center">
  <img src="graphics/08a_update_where.png" width="720" alt="With WHERE customer_id = 2, a SELECT check finds Bola, her limit goes from 200 to 250, and one row changes; without WHERE, every customer gets 50 more and five rows change; inside BEGIN and ROLLBACK the damage is visible and then undone">
</p>

`DELETE` has the identical trap. The [DELETE documentation](https://www.sqlite.org/lang_delete.html) says "If the WHERE clause is not present, all records in the table are deleted":

In [10]:
%%sql
BEGIN;

DELETE FROM orders;

SELECT COUNT(*) AS orders_left
FROM orders;

ROLLBACK;

5 rows deleted


orders_left
0


Five orders deleted, zero left, and then all five back after `ROLLBACK`. Two habits cover both statements: write the `WHERE` first and check it with a `SELECT`, and when a change matters, run it inside `BEGIN`, look at the result, and only then `COMMIT`.

## 6. DELETE: removing rows

With a `WHERE`, `DELETE` removes only the matching rows. Efe was only here to demonstrate `INSERT`, so take her back out:

In [11]:
%%sql
DELETE FROM customers
WHERE customer_id = 5;

1 row deleted


"1 row deleted", and `customers` is back to four rows. `DELETE` removes rows and keeps the table; removing the table itself, structure and all, is `DROP TABLE`, which `08b` uses.

## 7. Deleting a row that other rows point to

Efe had no orders, so nothing else in the database cared that she left. Chidi is different: order 105 has `customer_id = 3`. `04 - JOINs` mentioned in passing that this setup "doesn't declare a foreign key constraint", and here's what that sentence was warning about. This runs inside a transaction, so it's safe:

In [12]:
%%sql
BEGIN;

DELETE FROM customers
WHERE customer_id = 3;

SELECT
    c.customer_name,
    SUM(o.amount) AS total_spent
FROM orders AS o
INNER JOIN customers AS c
    ON o.customer_id = c.customer_id
GROUP BY
    c.customer_name
ORDER BY
    c.customer_name;

SELECT SUM(amount) AS all_orders
FROM orders;

1 row deleted


customer_name,total_spent
Ada,100
Bola,170


all_orders
410


The delete succeeded, and the customer report now shows Ada at 100 and Bola at 170, a total of 270, while the plain sum of `orders` still says 410. Order 105 is still in `orders`, pointing at a customer who no longer exists, and the `INNER JOIN` drops any row without a match, so 140 of revenue disappears from one report while staying in another. An order like this, whose parent row is gone, is called an **orphan**, and `04`'s other join finds it:

In [13]:
%%sql
SELECT
    o.order_id,
    o.customer_id,
    o.amount
FROM orders AS o
LEFT JOIN customers AS c
    ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL;

order_id,customer_id,amount
105,3,140


Order 105, customer 3, amount 140. The transaction is still open, so roll it back and Chidi is back:

In [14]:
%%sql
ROLLBACK;

<p align="center">
  <img src="graphics/08a_orphaned_order.png" width="640" alt="Without an enforced foreign key, deleting Chidi leaves order 105 behind, so SUM over orders says 410 while the INNER JOIN report says 270; with REFERENCES and PRAGMA foreign_keys = ON, the delete is rejected and every report agrees">
</p>

The fix is a **foreign key**, which declares that `orders.customer_id` must match an existing `customers.customer_id`, so the database refuses the delete instead of leaving an orphan. Setting one up, and switching it on in SQLite, is where `08b` starts.

## 8. Try it yourself

Write each query in a new cell before looking at the [solutions notebook](<solutions/08a-changing-data-part-1-solutions.ipynb>), predicting the result first. Run anything that changes data between `BEGIN` and `ROLLBACK`, so the tables stay as they are.

1. **Three statements, one table.** Before running this, predict how many rows the `DELETE` removes, and what `orders` looks like at the end:
   ```sql
   BEGIN;

   INSERT INTO orders (
       order_id,
       customer_id,
       amount
   )
   VALUES (106, 4, 50);

   UPDATE orders
   SET amount = amount + 10
   WHERE customer_id = 1;

   DELETE FROM orders
   WHERE amount < 50;

   SELECT
       order_id,
       customer_id,
       amount
   FROM orders
   ORDER BY
       order_id;

   ROLLBACK;
   ```
2. **A raise for customers who've ordered.** Give every customer who has placed at least one order a credit limit 10 percent higher, leaving Dami unchanged, and show the result before rolling back. (A subquery from `05` decides who has ordered.)
3. **Keep a copy before you change anything.** Make a backup of `orders` called `orders_backup` with `CREATE TABLE orders_backup AS SELECT * FROM orders;`, then delete every order over 100 from `orders` (inside a transaction), and compare the two tables' row counts before rolling back.
4. **Stretch, on AdventureWorks.** Copy the products that have a list price into a temporary table, then lower the price of every product with no recorded color by 10 percent, inside a transaction. How many rows did the update change? Save this one for after section 9.

## 9. Scale it up: writing to real data, safely

Five orders let you predict every count by hand. Real tables are where a missing `WHERE` changes thirty thousand rows in a fraction of a second. This section runs the same statements against **AdventureWorks2022**, Microsoft's sample database for an invented bike company, on SQL Server, with one safety rule: **every write goes to a temporary copy**, never to AdventureWorks itself.

### Set up AdventureWorks

SQL Server is a separate program from this notebook, so it needs one setup step before the `%%tsql` cells can run:

- **In Google Colab**, the cell below installs SQL Server and AdventureWorks inside your Colab session for you. The first run takes a few minutes and prints each step as it goes. Colab gives you a fresh machine every session, so run it again whenever you come back to the notebook later.
- **On your own machine**, it checks that SQL Server is reachable and tells you what's missing if it isn't; the [Course Overview](<00-course-overview.ipynb>) covers installing SQL Server and AdventureWorks locally.

If you'd rather not set anything up, every `%%tsql` cell below keeps its saved result, so you can read the section without running it.

In [15]:
# Run this once before any %%tsql cell.
# In Google Colab it installs SQL Server and AdventureWorks into this session (about 3 to 5
# minutes the first time, and again in each new session, because Colab starts fresh).
# On your own machine it just checks that your SQL Server is reachable.
%setup_adventureworks

AdventureWorks2022 is ready, so every %%tsql cell in this notebook will run.


### A temporary copy to practise on

On SQL Server, a table whose name starts with `#` is a **temporary table**: it exists only for your session and disappears when the session ends. `SELECT ... INTO` creates one and fills it in a single statement:

In [16]:
%%tsql
SELECT
    SalesOrderID,
    CustomerID,
    OrderDate,
    TotalDue
INTO #order_copy
FROM Sales.SalesOrderHeader;

31465 rows copied


31,465 rows copied. Everything below changes `#order_copy`, and `Sales.SalesOrderHeader` is untouched.

### Check, then UPDATE

Say every 2014 order gets a 10 percent discount. Check the `WHERE` first:

In [17]:
%%tsql
SELECT COUNT(*) AS orders_to_change
FROM #order_copy
WHERE OrderDate >= '2014-01-01';

orders_to_change
11761


In [18]:
%%tsql
UPDATE #order_copy
SET TotalDue = TotalDue * 0.9
WHERE OrderDate >= '2014-01-01';

11761 rows updated


11,761 rows updated, the same number the `SELECT` promised. When those two numbers disagree, stop and find out why before you go further. On SQL Server, `@@ROWCOUNT` returns the count of the statement that just ran, the equivalent of SQLite's `changes()`, which is useful inside scripts that need to check their own work.

### The missing WHERE, at scale

Here's section 5's mistake on 31,465 rows, safely inside a transaction. SQL Server writes `BEGIN TRANSACTION`, and `ROLLBACK` works the same way:

In [19]:
%%tsql
BEGIN TRANSACTION;

UPDATE #order_copy
SET TotalDue = 0;

SELECT COUNT(*) AS zero_orders
FROM #order_copy
WHERE TotalDue = 0;

ROLLBACK;

31465 rows updated


zero_orders
31465


Every order zeroed in one statement, and then restored. Check that the rollback really did undo it:

In [20]:
%%tsql
SELECT COUNT(*) AS zero_orders_after_rollback
FROM #order_copy
WHERE TotalDue = 0;

zero_orders_after_rollback
0


### DELETE with a WHERE

Remove the tiny orders, under $5, from the copy:

In [21]:
%%tsql
DELETE FROM #order_copy
WHERE TotalDue < 5;

530 rows deleted


530 rows deleted, which is more than the 237 orders under $5 in the original table. The difference is the 2014 discount from earlier in this section: it pushed another 293 orders below $5 in the copy, and the `DELETE` saw the table the `UPDATE` left behind, the same thing exercise 1 asks you to predict. A real clean-up would keep a backup first (exercise 3 shows how), run inside a transaction until the count looks right, and check the count against what you expected before committing.

### Orphans in a database that won't allow them

Section 7's orphan finder, run against the real tables:

In [22]:
%%tsql
SELECT COUNT(*) AS orphaned_orders
FROM Sales.SalesOrderHeader AS h
LEFT JOIN Sales.Customer AS c
    ON h.CustomerID = c.CustomerID
WHERE c.CustomerID IS NULL;

orphaned_orders
0


Zero, and not by luck. AdventureWorks declares a foreign key from every order to its customer, so the database refuses any delete that would leave an order behind. `08b` shows that rule being declared, and what happens when you try to break it.

## Before you change data

- Write the `WHERE` before the `SET`, and run it as a `SELECT` first.
- Read the row count after every write; zero rows is not an error, and every row is not a warning.
- When a change matters, run it inside `BEGIN`, check the result, and only then `COMMIT`.
- Name the columns in every `INSERT`.
- Before deleting a row, ask what points to it.
- Keep a backup, or practise on a copy, before a large change.

## Go deeper

- **[INSERT](https://www.sqlite.org/lang_insert.html)**, **[UPDATE](https://www.sqlite.org/lang_update.html)**, **[DELETE](https://www.sqlite.org/lang_delete.html)**, and **[Transactions](https://www.sqlite.org/lang_transaction.html)**, SQLite. Short references that spell out the no-`WHERE` behavior in plain words.
- **[INSERT](https://learn.microsoft.com/en-us/sql/t-sql/statements/insert-transact-sql)**, **[UPDATE](https://learn.microsoft.com/en-us/sql/t-sql/queries/update-transact-sql)**, **[DELETE](https://learn.microsoft.com/en-us/sql/t-sql/statements/delete-transact-sql)**, and **[BEGIN TRANSACTION](https://learn.microsoft.com/en-us/sql/t-sql/language-elements/begin-transaction-transact-sql)**, Microsoft Learn. The SQL Server versions, including the `OUTPUT` clause that shows exactly which rows a write changed.
- **[PostgreSQL tutorial: Updates](https://www.postgresql.org/docs/current/tutorial-update.html)** and **[Deletions](https://www.postgresql.org/docs/current/tutorial-delete.html)**. Two short pages with the same warning about a missing `WHERE`.
- **Practice:** [SQLBolt](https://sqlbolt.com/lesson/inserting_rows) lessons 13 to 15 cover inserting, updating, and deleting, with exercises checked in the browser.

For interview questions on this topic with model answers, see the [study guide](<study-guides/08a-changing-data-part-1.md>).

## What's next

This notebook changed rows inside rules somebody else wrote. `08b - Changing Data, Part 2` writes the rules: `CREATE TABLE` line by line, the constraints that stop bad rows at the door, the foreign key that would have saved Chidi's order, and `ALTER TABLE` for changing a table that's already full of data.

---

[← 07 - Window Functions, Part 2](<07-window-functions-part-2.ipynb>) · [Course Overview](<00-course-overview.ipynb>) · [08b - Changing Data, Part 2 →](<08b-changing-data-part-2.ipynb>)

*SQL_Tutorial* is written and maintained by Samuel Shaibu as part of *All About Data & More*. Licensed under [MIT](LICENSE).